# EDA of the ME4OH dataset filtered to the North Atlantic Basin


In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from Data.load_sst_ofam3_data import NA_DATA_PATH
from helper_functions.plotting_functions import (
    plot_compare_ff_insitu,
    plot_ohc_map,
    plot_sst_map,
)

In [ ]:
COUNTRIES_URL = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
WORLD = gpd.read_file(COUNTRIES_URL).rename({"ADMIN": "name"}, axis="columns")
regions = ['Southern Europe', 'Northern Africa', 'Western Africa', 'Western Europe', 'Northern Europe', 'Central America', 'South America', 'Northern America', 'Caribbean']
NA = WORLD[WORLD['SUBREGION'].isin(regions)]

## EDA - OHC

In [ ]:
df = pd.read_csv("Data/ME4OH_EN411_OFAM3/NA_1993_2014.csv")

# Convert dates to datetime, save year month and year indication columns (aids plotting)
df['Date'] = pd.to_datetime(df['Date'])
df['year_month'] = df['Date'].dt.to_period('M')
df['year'] = df['Date'].dt.to_period('Y')
df

### Plot OHC in latitude and longitude

In [ ]:
month_list = pd.date_range('1993-1-1','2014-12-31', freq='MS').strftime('%Y-%m').tolist()
num_plots = 10
plot_months = [month_list[i] for i in range(0, len(month_list), len(month_list)//num_plots)]

In [ ]:
for n,g in df.groupby('year_month'):
    if str(n) in plot_months:
        plot_ohc_map(g, n, min(df.OHC), max(df.OHC), area="NA")

In [ ]:
for n,g in df.groupby('year'):
    plot_ohc_map(g, n, min(df.OHC), max(df.OHC), area="NA")

### Plot by platform (OHC dataset)

In [ ]:
# Filter by platforms
platforms = {
    "Argo": df.loc[df['Project name'].str.contains('ARGO')].sort_values('Date').reset_index(drop=True),
    "WOD": df.loc[df['Project name'].str.contains('WOD')].sort_values('Date').reset_index(drop=True),
    "GTSPP": df.loc[df['Project name'].str.contains('GTSPP')].sort_values('Date').reset_index(drop=True),
    "ABSO":  df.loc[df['Project name'].str.contains('ASBO')].sort_values('Date').reset_index(drop=True),
}

In [ ]:
colours = [ "#631ff3", "#ffb000", "#fe5100", "#1589e8", "#dc267f"]  # IBM colourblind palette
year_list = pd.date_range('1993-1-1','2014-12-31', freq='YS').strftime('%Y').tolist()

In [ ]:
for year in year_list:
    _, ax = plt.subplots(figsize=(12,6))
    NA.plot(ax=ax, color='lightgray', edgecolor='gray', linewidth=0.75)  # Add countries
    for i, (name, data) in enumerate(platforms.items()):
        year_data = data.loc[data["year"] == year]
        ax.scatter(year_data.Longitude, year_data.Latitude, s=1, label=name, c=colours[i])
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_facecolor("lightblue")
    ax.set_xlim([-80, 0])
    ax.set_ylim([0, 60])
    plt.legend(bbox_to_anchor=(1,1))
    plt.title(year)
    plt.savefig(f"images/platforms_NA_by_year/platforms_NA_{year}.png")
    plt.show()
    plt.close()

## EDA - SST

Note: Assumptions as of 30/06/26
- OFAM3 data is stored for Thursdays across the 1979-2014 time period.
- Pending confirmation from the ME4OH team, I have assumed that the data is the full-field SST values generated by the simulator model (OFAM3) averaged across a calendar week

### Example visualisation

In [ ]:
date = '1979-01-04'
df = pd.read_csv(f"Data/OFAM3/world/{date}.csv")
# df['Date'] = pd.to_datetime('1979-01-04')
plot_sst_map(df, date, min(df.SST), max(df.SST))

In [ ]:
date = '2014-06-19'
df = pd.read_csv(f"{NA_DATA_PATH}{date}.csv")
plot_sst_map(df, date, min(df.SST), max(df.SST), area='NA')

In [ ]:
# ensure datetime and ISO week/year columns exist
na_df = pd.read_csv("Data/ME4OH_EN411_OFAM3/SST_NA_1993_2014.csv")
na_df['Date'] = pd.to_datetime(na_df['Date'])
na_df['iso_year'] = na_df['Date'].dt.isocalendar().year
na_df['iso_week'] = na_df['Date'].dt.isocalendar().week
na_df

In [ ]:
ref_date = pd.Timestamp("2014-06-19")
ref_year, ref_week = ref_date.isocalendar()[:2]

# keep only the same ISO week/year
week_df = na_df[
    (na_df['iso_year'] == ref_year) &
    (na_df['iso_week'] == ref_week)
].copy()

In [ ]:
plot_sst_map(week_df, f"Data from week containing: {ref_date.date()}", min(week_df.SST), max(week_df.SST), area="NA", background=True)

In [ ]:
date = '2014-06-19'
df = pd.read_csv(f"{NA_DATA_PATH}{date}.csv")

In [ ]:
plot_compare_ff_insitu(df, week_df, area="NA", title=f"Data from week containing: {ref_date.date()}")

In [ ]:
# How many sensors per year?
na_df

In [ ]:
na_df.groupby('iso_year').size()

In [ ]:
na_df.groupby(['iso_year', 'iso_week']).size()

In [ ]:
np.mean(na_df.groupby(['iso_year', 'iso_week']).size())